# `defect_classify_into_lines()`

**This is not a general-purpose point-classification, clustering, or nearest-neighbor algorithm. It is specialized for the half-grid defect-index format produced by `defect_detect()`.**

The analysis function `nematics3d.defect_classify_into_lines()` connects detected defect plaquettes into ordered disclination-line trajectories. It returns one `DisclinationLine` object for every extracted graph trail.

This function does not detect defects from a director or $Q$-tensor field. Its input is the half-grid defect-index array already produced by `defect_detect()` or an equivalent calculation. The classifier determines which defect plaquettes are geometrically adjacent, separates disconnected structures, orders the points along each trail, and unwraps trails that cross periodic boundaries.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell imports `NumPy` and `Nematics3D`.


In [1]:
import numpy as np
import nematics3d as n3d


## Minimal example: separate two defect lines

The input below contains two disconnected chains of defect plaquettes. The rows are grouped only to make the example easy to read; the classifier determines connectivity from the coordinates rather than from a supplied line label.


In [2]:
first_chain = np.array([[x, 0.5, 0.5] for x in range(4)])
second_chain = np.array([[x, 10.5, 10.5] for x in range(3)])
defect_indices = np.concatenate([first_chain, second_chain])

lines = n3d.defect_classify_into_lines(defect_indices)
print("number of lines:", len(lines))
for index, line in enumerate(lines):
    print(f"line {index}:\n", line.raw_defect_indices)


number of lines: 2
line 0:
 [[0.  0.5 0.5]
 [1.  0.5 0.5]
 [2.  0.5 0.5]
 [3.  0.5 0.5]]
line 1:
 [[ 0.  10.5 10.5]
 [ 1.  10.5 10.5]
 [ 2.  10.5 10.5]]


## Inputs and outputs

The public signature is:

```python
defect_classify_into_lines(
    defect_indices,
    box_size_periodic=np.inf,
    grid_offset=None,
    grid_transform=GRID_TRANSFORM_IDENTITY,
)
```

### Accepted defect-index representation

`defect_indices` must be a real, finite array with shape `(N, 3)`. Every row is the center of one defective lattice plaquette and must have exactly one integer coordinate and two half-integer coordinates:

| Example | Plaquette orientation |
| --- | --- |
| `(2.0, 3.5, 4.5)` | normal to $x$ |
| `(2.5, 3.0, 4.5)` | normal to $y$ |
| `(2.5, 3.5, 4.0)` | normal to $z$ |

Integer and half-integer values are snapped within a tolerance of `5e-10`. Off-lattice coordinates, invalid parity, duplicate canonical coordinates, non-finite values, and shapes other than `(N, 3)` are rejected. Empty input is accepted and returns an empty list.

### Periodic box and coordinate mapping

| Argument | Meaning |
| --- | --- |
| `box_size_periodic=np.inf` | Treat every axis as non-periodic |
| one finite scalar | Use one periodic lattice size for all axes |
| three finite or infinite values | Configure the $x$, $y$, and $z$ axes separately |
| `grid_transform` | Apply a $3\times3$ linear map from lattice-index coordinates to physical coordinates |
| `grid_offset` | Translate the transformed coordinates by one three-component vector |

Finite periodic sizes must be positive integers because `defect_indices` live in lattice-index space. Positive infinity marks a non-periodic axis. `grid_transform` and `grid_offset` do not change graph connectivity; they are carried into each returned line and used to calculate physical coordinates.

### Returned lines

The function returns `list[DisclinationLine]`. Important fields on each line include:

| Field | Meaning |
| --- | --- |
| `raw_defect_indices` | ordered and periodically unwrapped lattice-index trajectory |
| `calc_defect_coords` | corresponding physical coordinates after the grid transform and offset |
| `raw_box_size_periodic_index` | normalized three-axis periodic size in lattice-index space |
| `calc_defect_num` | number of stored defect points |

A non-empty input can still return an empty list when every defect is isolated and no graph edge exists. At a branch, a physical junction may appear more than once because the classifier returns edge-consuming trails rather than forcing a branched graph into one simple path.


## Examples


### Connect a line across a periodic boundary

The two points below are separated when $x$ is open. With an $x$ period of 4, indices 3 and 0 are neighboring periodic images, so the classifier creates one continuous line and unwraps the jump.


In [3]:
boundary_defects = np.array([[0.0, 0.5, 0.5], [3.0, 0.5, 0.5]])

open_lines = n3d.defect_classify_into_lines(boundary_defects)
periodic_lines = n3d.defect_classify_into_lines(
    boundary_defects,
    box_size_periodic=[4.0, np.inf, np.inf],
)
print("open boundaries:", len(open_lines), "lines")
print("periodic boundary:", periodic_lines[0].raw_defect_indices)


open boundaries: 0 lines
periodic boundary: [[ 0.   0.5  0.5]
 [-1.   0.5  0.5]]


### Map lattice indices to physical coordinates

Connectivity is evaluated in lattice-index space. The returned `DisclinationLine` separately calculates physical coordinates using `grid_transform` followed by `grid_offset`.


In [4]:
transform = np.diag([2.0, 1.0, 0.5])
offset = np.array([10.0, -1.0, 3.0])
mapped_line = n3d.defect_classify_into_lines(
    first_chain,
    grid_transform=transform,
    grid_offset=offset,
)[0]
print("index coordinates:\n", mapped_line.raw_defect_indices)
print("physical coordinates:\n", mapped_line.calc_defect_coords)


index coordinates:
 [[0.  0.5 0.5]
 [1.  0.5 0.5]
 [2.  0.5 0.5]
 [3.  0.5 0.5]]
physical coordinates:
 [[10.   -0.5   3.25]
 [12.   -0.5   3.25]
 [14.   -0.5   3.25]
 [16.   -0.5   3.25]]


## Special examples


### A branched defect graph

A branch is not representable as one simple line without revisiting a junction. The classifier therefore consumes every graph edge exactly once and may repeat the central defect index inside the returned trail. This preserves connectivity rather than silently dropping one branch.


In [5]:
branched_defects = np.array(
    [
        [0.0, 0.5, 0.5],
        [-1.0, 0.5, 0.5],
        [1.0, 0.5, 0.5],
        [0.5, 1.0, 0.5],
    ]
)
branch_lines = n3d.defect_classify_into_lines(branched_defects)
for line in branch_lines:
    print(line.raw_defect_indices)


[[ 0.   0.5  0.5]
 [ 1.   0.5  0.5]
 [ 0.5  1.   0.5]
 [ 0.   0.5  0.5]
 [-1.   0.5  0.5]]


## Details

### From plaquettes to graph trails

Each valid defect index represents the center of a defective plaquette, or equivalently a link of the dual cubic lattice passing normally through that plaquette. Two defect indices are neighbors only when their dual-lattice links share an endpoint. This lattice rule—not Euclidean distance alone—is what defines continuation along a disclination line.

A dual-lattice link has two endpoints. At either endpoint, six links meet: the current link, one collinear continuation, and four perpendicular links. After excluding the current link, there are therefore five possible continuations at each endpoint. The endpoint sets do not overlap, giving exactly

$$2\times(1+4)=10$$

legal neighbors: two collinear neighbors and eight perpendicular neighbors. No diagonal link, link that merely passes nearby, or link that fails to share one of these two endpoints is legal.

For example, a defect index $(i, j+\tfrac12, k+\tfrac12)$ has an integer x coordinate and represents a yz plaquette, hence a dual link parallel to x. In doubled integer coordinates its ten allowed offsets are

$$
(\pm2,0,0),\qquad
(\pm1,\pm1,0),\qquad
(\pm1,0,\pm1).
$$

The first pair are the two collinear x-directed continuations. The other eight reach the four y- or z-directed links incident at each of the two endpoints. Permuting x, y, and z gives the corresponding ten offsets for the other two plaquette orientations. Only candidates actually present in the supplied defect-index array become undirected graph edges. The classifier then starts at odd-degree nodes when possible and consumes every edge into maximal ordered trails. Components containing no edge produce no line.

### Doubled integer coordinates

Internally, every defect coordinate is multiplied by two. Integer and half-integer lattice positions therefore become exact integers, so periodic wrapping and neighbor matching do not depend on floating-point equality. The three integer coordinates are packed into one collision-free integer key for lookup.



## Possible issues

### Isolated defects

A single isolated defect plaquette has no edge and is therefore not returned as a one-point `DisclinationLine`. Check the input or preserve isolated indices separately if they matter to the application.

### Branches are trails, not separate connected components

The output partitions graph edges into ordered trails. It does not promise exactly one line object per connected component, and junction nodes can occur in more than one position or trail.

### Periodic duplicates

Two input coordinates that become identical after periodic wrapping are ambiguous duplicate nodes and are rejected. Finite periodic sizes must also be integer-valued in lattice-index space.

### Output ordering

Line and traversal ordering is deterministic for a fixed implementation and input, but it should not be treated as a physical ranking. Sort returned lines explicitly when an application needs length, position, or another semantic order.


## Logging and progress information

Set `log_level=logging.DEBUG` to report how many defects and edges were classified, how many trails were produced, and how long graph construction and traversal took. Nested periodic unwrapping also reports the normalized periodic box for each returned trail.


In [6]:
import logging

debug_lines = n3d.defect_classify_into_lines(
    defect_indices,
    log_level=logging.DEBUG,
)


[DEBUG]
    Function `defect_classify_into_lines` STARTED in program `ipykernel_launcher.py`
[DEBUG]
    <defect_classify_into_lines> 
    Classified 7 defects through 5 edges into 2 trails in 0.000 seconds.
[DEBUG]
        <unwrap_trajectory> 
        Unwrapping 4 point(s); box_size_periodic=[inf, inf, inf], is_reverse=False, is_start_in_box=False.
[DEBUG]
        <unwrap_trajectory> 
        Unwrapping 3 point(s); box_size_periodic=[inf, inf, inf], is_reverse=False, is_start_in_box=False.
[DEBUG]
        <DisclinationLine.__init__> 
        Disclination line 'disclination line' is of kind 'seg'
[DEBUG]
        <DisclinationLine.__init__> 
        Disclination line 'disclination line' is of kind 'seg'
[DEBUG]
    Function `defect_classify_into_lines` FINISHED in program `ipykernel_launcher.py`. Elapsed time: 0.002 seconds.


## Where `defect_classify_into_lines()` is used

Most users encounter this operation through [`QFieldObject.act_lines_classify()`](../../classes/QFieldObject/act_lines_classify.ipynb), which classifies the defect indices stored by a high-level $Q$-field object. Call the low-level function directly when defect indices are already available as a standalone array, when periodic lattice sizes must be supplied explicitly, or when custom grid transforms and offsets are needed without constructing a `QFieldObject`.


## Useful Links

### Referenced in this tutorial

- [`defect_classify_into_lines()` source](../../../src/nematics3d/analysis/disclination/classification.py) — implements vectorized edge construction and trail extraction.
- [`as_defect_index()` source](../../../src/nematics3d/datatypes/defect_index.py) — defines the accepted half-grid defect-index representation.
- [`as_box_size_periodic()` source](../../../src/nematics3d/datatypes/box_size_periodic.py) — validates periodic box information.
- [`unwrap_trajectory()` tutorial](../../grid/periodic/unwrap_trajectory.ipynb) — explains how periodic trails are made continuous.
- [`DisclinationLine` source](../../../src/nematics3d/classes/disclination_line.py) — defines the returned line object.

### Going deeper

- [Defect detection](defect_detect.ipynb) — obtains half-grid defect indices from a director field.
- [`QFieldObject.act_lines_classify()`](../../classes/QFieldObject/act_lines_classify.ipynb) — performs classification through the high-level $Q$-field workflow.
- [`QFieldObject.act_lines_smooth()`](../../classes/QFieldObject/act_lines_smooth.ipynb) — smooths classified disclination lines for downstream analysis and visualization.
